In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from tokenizers import Tokenizer
from torch.nn import CrossEntropyLoss
from tqdm import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, PreTrainedTokenizerFast

PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "model" / "GenNA"
TOKENIZER_JSON_PATH = PROJECT_ROOT / "configs" / "tokenizer.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
FLASH_ATTN_AVAILABLE = DEVICE.type == "cuda" and importlib.util.find_spec("flash_attn") is not None
MAX_LENGTH = 4096

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Required path not found: {MODEL_DIR}")
if not TOKENIZER_JSON_PATH.exists():
    raise FileNotFoundError(f"Required path not found: {TOKENIZER_JSON_PATH}")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.linewidth": 1.2,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.unicode_minus": False,
})


def build_tokenizer(tokenizer_path: Path) -> PreTrainedTokenizerFast:
    tokenizer_obj = Tokenizer.from_file(str(tokenizer_path))
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_obj,
        unk_token="<unk>",
        pad_token="<pad>",
        eos_token="<eos>",
        model_max_length=MAX_LENGTH,
    )
    return tokenizer


def load_model(model_dir: Path, device: torch.device, dtype: torch.dtype):
    config = AutoConfig.from_pretrained(model_dir)
    model_kwargs = {
        "config": config,
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    if FLASH_ATTN_AVAILABLE:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs).to(device)
    model.eval()
    return model


tokenizer = build_tokenizer(TOKENIZER_JSON_PATH)
model = load_model(MODEL_DIR, DEVICE, DTYPE)
print(f"Model loaded on {DEVICE}")


In [ ]:
def calc_perplexity_batch(model, tokenizer, texts, device=None, batch_size=8):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Calculating PPL"):
        batch_texts = texts[i:i + batch_size]
        encodings = tokenizer(
            batch_texts,
            padding=True,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
        )

        input_ids = encodings.input_ids.to(device)
        attention_mask = encodings.attention_mask.to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.float()

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        shift_mask = attention_mask[..., 1:].contiguous()

        loss_fct = CrossEntropyLoss(reduction="none")
        vocab_size = shift_logits.size(-1)
        flat_loss = loss_fct(shift_logits.view(-1, vocab_size), shift_labels.view(-1))
        per_token_loss = flat_loss.view(shift_labels.size())
        masked_loss = per_token_loss * shift_mask

        sum_loss = masked_loss.sum(dim=1)
        num_valid_tokens = shift_mask.sum(dim=1)
        mean_loss = sum_loss / num_valid_tokens.clamp(min=1)
        ppls = torch.exp(mean_loss)

        for ppl, count in zip(ppls.cpu().tolist(), num_valid_tokens.cpu().tolist()):
            results.append({
                "perplexity": float(ppl),
                "n_tokens": int(count),
            })

        del input_ids, attention_mask, logits, outputs, shift_logits, shift_labels, flat_loss
        if device.type == "cuda":
            torch.cuda.empty_cache()

    return results


In [ ]:
def extract_gene_region(full_text, start_tag="<gene>", end_tag="</gene>"):
    start_idx = full_text.find(start_tag)
    end_idx = full_text.find(end_tag)
    if start_idx == -1 or end_idx == -1:
        raise ValueError("Could not find <gene> ... </gene> in the text.")
    gene_start = start_idx + len(start_tag)
    gene_end = end_idx
    prefix = full_text[:gene_start]
    gene_seq = full_text[gene_start:gene_end]
    suffix = full_text[gene_end:] 
    return prefix, gene_seq, suffix

def index_nt_sites(gene_seq, allowed_bases=("A","C","G","T","a","c","g","t")):
    sites = []
    inside_tag = False
    nt_index = 0
    for i, ch in enumerate(gene_seq):
        if ch == "<":
            inside_tag = True
            continue
        if ch == ">":
            inside_tag = False
            continue
        if inside_tag:
            continue
        if ch in allowed_bases:
            sites.append({"nt_index": nt_index, "abs_index": i, "base": ch})
            nt_index += 1
    return sites

def generate_single_nt_mutants_strict(prefix, gene_seq, suffix, allowed_bases=("A","C","G","T","a","c","g","t"), max_positions=None, include_deletion=True):
    nt_sites = index_nt_sites(gene_seq, allowed_bases=allowed_bases)
    mutants = []
    gene_list = list(gene_seq)
    upper_candidates = {b: [c for c in ["A","C","G","T"] if c != b] for b in ["A","C","G","T"]}
    lower_candidates = {b: [c for c in ["a","c","g","t"] if c != b] for b in ["a","c","g","t"]}

    for site in nt_sites:
        k = site["nt_index"]
        i = site["abs_index"]
        orig_base = site["base"]

        if max_positions is not None and k >= max_positions:
            break

        candidates = upper_candidates.get(orig_base, []) if orig_base.isupper() else lower_candidates.get(orig_base, [])

        for new_base in candidates:
            mutated_gene_list = gene_list[:]
            mutated_gene_list[i] = new_base
            mut_text = prefix + "".join(mutated_gene_list) + suffix
            mutants.append({
                "pos_in_gene_nt": k,
                "abs_index_in_gene": i,
                "ref_base": orig_base,
                "mut_base": new_base,
                "mutation_type": "substitution",
                "mut_text": mut_text,
            })

        if include_deletion:
            mutated_gene_list = gene_list[:]
            del mutated_gene_list[i]
            mut_text = prefix + "".join(mutated_gene_list) + suffix
            mutants.append({
                "pos_in_gene_nt": k,
                "abs_index_in_gene": i,
                "ref_base": orig_base,
                "mut_base": "DEL",
                "mutation_type": "deletion",
                "mut_text": mut_text,
            })
    return mutants


In [ ]:
def scan_single_nt_effects_batch(full_text, tokenizer, model, device, max_positions=None, include_deletion=True, batch_size=8):
    wt_res_list = calc_perplexity_batch(model, tokenizer, [full_text], device=device, batch_size=1)
    wt_res = wt_res_list[0]
    wt_ppl = wt_res["perplexity"]
    
    print(f"WT Perplexity: {wt_ppl:.4f}")

    prefix, gene_seq, suffix = extract_gene_region(full_text)
    mutants = generate_single_nt_mutants_strict(
        prefix, gene_seq, suffix,
        max_positions=max_positions,
        include_deletion=include_deletion
    )
    
    print(f"Generated {len(mutants)} mutants. Calculating PPL in batches of {batch_size}...")

    mutant_texts = [m["mut_text"] for m in mutants]

    ppl_results = calc_perplexity_batch(model, tokenizer, mutant_texts, device=device, batch_size=batch_size)

    records = []
    for m, res in zip(mutants, ppl_results):
        records.append({
            "pos_in_gene_nt": m["pos_in_gene_nt"],
            "abs_index_in_gene": m["abs_index_in_gene"],
            "ref_base": m["ref_base"],
            "mut_base": m["mut_base"],
            "mutation_type": m["mutation_type"],
            "wt_ppl": wt_ppl,
            "mut_ppl": res["perplexity"],
            "delta_ppl": res["perplexity"] - wt_ppl,
            "n_tokens_mut": res["n_tokens"],
        })

    df = pd.DataFrame.from_records(records).sort_values(["pos_in_gene_nt", "mutation_type"]).reset_index(drop=True)
    return df, wt_res


In [ ]:
text = "RNA, Spodoptera litura, LOC111353161, glutaminase liver isoform, mitochondrial isoform X1<seq><gene>CATTTGATTGTTTTGTAATATATTGTATACTAGATAATATATGATGGCATAGGTTCCCGTCTATTACTAATTTGGCACTCCCAGATGTAGTATATTTATAAACGGGAGACAGAAGGAGA<CDS>ATGGCTCAGTGTGTCGTGCGCGGAAGTGTCGGGCATTTCGCACGATATTCGGCTCAGTTCTCGCAATTCACGGCGCGTACGCATCCCTCGCAGCCGTCCCGCAACTCTGGCATGCGCTACTTGTACAACAGTGGCTTCGAAGAATTTATGATGGAATGTGATAACAGTTTTGAGAAATGGCGCAATTCTGCCCCGCATTTCTATTCAGTCCAGTCCCTCGACTTGAGCAGCAAAACCCGACAGTACTCGTTCGTGCATAACATCGACCCCATATCGGTGCGCGAAGGACCGCACAAGAACGCTGACGATGTTTTATTTGACATGTTTAAGAATGAAGACACGGGTCTCCTTCCCGTTGGAAAGTTTTTGGCCGCCCTGAGGACCACTGGCATAAGGAAAAATGATAATCGTGTAAAAGAAGTCATGCATAATCTATACAAGGTGCACAAAGAGTCCAACTTTGAAGGCGGTTCACCGGAGACATTGAAACTAGACAGGAAGACCTTCAAAGAGGTGATCGCACCTAACATCGTCCTCATCACGAGGGCGTTCCGCAGTCAATTCGTAATTCCAGACTTTCAAGATTTTATCAAAGACGTGGAAGAAATGTACTGGGCTGCTAAAAGTAACACTGACGGTAAAGTGGCCAGCTACATCCCGCAGCTCGCTAGGGCCAGCTCAGAGAACTGGGGTGTCAGCATTTGTACTATAGACGGTCAACGATTTTCTATCGGTGATGTGACGGTTCCATTCACTCTACAATCGTGCAGCAAACCACTCACATACGCGATGGCACTTGAAACTCTAGGACCCGACGTGGTACACAAATATGTTGGCACGGAGCCCAGCGGCCGCAACTTCAATGAACTCGTTCTGGATTATAACATGAAGCCACATAACCCGATGATAAACGCGGGAGCAATTCTAGTATGCTCATTATTGAAGACCTTGGACCGGCCCGAGATGACTCTCGCTGAAAAGTTTGATTACGTCATGTCGTTCTTTAGTCGGCTTGCAGGCAACGAAAGCCTTGGTTTCAACAATGCAGTATTTTTATCTGAACGCGAAGCTGCCGACAGAAACTATGCACTTGGCTTCTACATGCGAGAGCATAAATGTTATCCGGAAAAAACTAATTTCCGTGAATGTATGGACTTCTATTTCCAGTGTTGCTCAATGGAAGCCAACTGCGACATCATGTCTATTATGGCTGCTACTCTGGCCAATGGTGGTATCTGTCCCATCACAGATGAAAAGGTACTACGGCCCGACTCGGTACGCAACGTGCTTTCTCTGATGCATAGCTGTGGAATGTACGACTACTCAGGACAGTTTGCATTCAAAGTGGGACTTCCTGCAAAGTCTGGCGTATCTGGTGCTATGCTCATAGTTGTCCCCAATGTTATGGGTATTTGCACATGGTCGCCACCTTTGGACCCTCTGGGAAACAGCTGCCGAGGCTTGCAATTCTGTGATGACATGATTAAGAGGTTCAACTTCCACAAGTACGATAACATTCGTTACGCCAGCCACAAGAAGGATCCTCGTCGGTACAAGTTCGAGTCTGCGGGTCTCAACATTGTCAACCTCTTGTTCTCGGCTGCTAGTGGTGATCTTAGCGCTCTACGACGCCACCACCTCTCCGGTATGGACATGACTCTGTCGGACTACGACGGTCGCACTGCGCTACATCTGGCAGCTGCAGAGGGTCACCTATCCTGCGTAGACTTCCTGCTCGCTCAGTGCGGAGTTCCACACGACCCTCGCGATCGCTGGGGCAGCCGACCACTTAACGAGGCCGAGACCTTCGGACACACCGCTGTCGTACAATATTTGAAGGAATGGGAGCAGACGCACCCTACCGACAAAACCGTGGGCGCAGCTGCAGGCGCGTCAGATGTCGATGACATATTGAAAGTTAAACCAGACGCAGATGACGCTGTTAAAGACCCCAGCATCCAGGAGATTGTATCCAGGAATGGAGACAAAGTGCAATAG</CDS>AATCTCTACTGGGATGATGTAATGAGATGATCTCGTGACTATGCGTCACAAAATAGTTTTTGAAGGTGACTCCGCATTGTGTGCAGATTATGTTTCTTTCTGTAGCAGTTATGTGAACATAAGAAAAGTTGCTAATAGAAATAAAAACGCATGATTCATAGAATGCAGAGCTGATCTTAAAGGAAAGAAAATCAAGTCCTACGTATCTATTTGTAGATACGGTTAAGGGTATGTGTTATCGCCCAAAACATATTTATTTAAGTAGAAAATATTGCATTTTACGTTAGGACAGCTTTGGCCGTGAATATTTTCAAATGTCCTCATGAAATTATACCTGGCTTTTAGAAGGTATTTTATACCGATATTTACAAAATAACTCTACAATTATGAGAATTTGTGTACATGAGTATGGTTCGAAGAAGTAAATGCTTTGATATGGAAGTTATTCAGCACTAAGCTCAAGGGTGTGATATTAAAGTAACATGGAAGCACAAGGCAACATCTGGTGAGGAGTGAACAACGGGAGAGATGCTTAGAGAAATTAACTAGTCTGCTGAGAACTTGAGAAGATTGGCTAAACAAAACTGTGTGCATTTTGTGTGAAATGTAATCTCTTTCATAGATTATTTTAAATGTGAAGATTTTATGCAATATGTATTCATAAGTAGGTAGTGTGTAGCTACTGATTTTAGAGAAATTTATTAGATAAAATACATAAATAGGCATGTACATTAACCATGGTATCTCAGTGTTAATCTCTCTACTAATCGTCTATCTTTAAGCATCTGATTGTAATAAAAATAACTCCCAAAACTGTATTTTAACATTCCATATGATATAAATGACAGGCTTGCTGTACCAGGAACAATCAAGTGTTTATTCTAGAAGAATATAAAAACTTGAAGGAATGTTAAAAATTATTAATTAAATAATAATGTTAATAAATATAAGTTATCACTTATATTCTGACAATAAATAGGAGCTATTCTATGCCATTATTGATGTTGTCCAAATGTTTGTCAATTAAAT</gene></seq><eos>"

df, wt_info = scan_single_nt_effects_batch(
    text,
    tokenizer,
    model,
    DEVICE,
    max_positions=None,
    include_deletion=True,
    batch_size=32
)


In [ ]:
display(df)


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['axes.linewidth'] = 1.2      # 统一边框粗细
plt.rcParams['xtick.major.width'] = 1.2   # 统一刻度粗细
plt.rcParams['ytick.major.width'] = 1.2
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'

def parse_gene_structure_exact(full_text):
    s_tag = "<gene>"
    e_tag = "</gene>"
    s_idx = full_text.find(s_tag)
    e_idx = full_text.find(e_tag)
    if s_idx == -1 or e_idx == -1: return len(full_text), 0, 0
    
    gene_content = full_text[s_idx + len(s_tag) : e_idx]
    pure_len = 0
    cds_start, cds_end = None, None
    
    i = 0
    while i < len(gene_content):
        if gene_content[i] == "<":
            j = gene_content.find(">", i)
            if j != -1:
                tag = gene_content[i : j+1]
                if tag == "<CDS>": cds_start = pure_len
                elif tag == "</CDS>": cds_end = pure_len
                i = j + 1
                continue
        if gene_content[i].strip(): pure_len += 1
        i += 1
    return pure_len, cds_start, cds_end

def plot_mutation_heatmap_final_polished(df, full_text):
    total_len, cds_start, cds_end = parse_gene_structure_exact(full_text)
    if cds_start is None: cds_start, cds_end = 100, 200

    max_scan_pos = df["pos_in_gene_nt"].max()
    display_limit = min(total_len, max_scan_pos + 1)
    tmp = df[df["pos_in_gene_nt"] < display_limit].copy()
    
    bases = ["A", "C", "G", "T", "DEL"]
    matrix = tmp.pivot_table(
        index="mut_base", columns="pos_in_gene_nt", values="delta_ppl"
    ).reindex(bases)

    mask = np.zeros_like(matrix, dtype=bool)
    ref_series = tmp.groupby("pos_in_gene_nt")["ref_base"].first()
    for col_pos in matrix.columns:
        if col_pos in ref_series.index:
            ref_b = ref_series[col_pos]
            if ref_b in bases:
                row_idx = bases.index(ref_b)
                if col_pos < matrix.shape[1]:
                    col_idx = matrix.columns.get_loc(col_pos)
                    mask[row_idx, col_idx] = True

    fig = plt.figure(figsize=(15, 6.5), dpi=300) 
    gs = gridspec.GridSpec(2, 2, width_ratios=[40, 1], height_ratios=[0.5, 10], 
                           wspace=0.015, hspace=0.03)

    ax_struc = fig.add_subplot(gs[0, 0])
    ax_heat = fig.add_subplot(gs[1, 0])
    ax_cbar = fig.add_subplot(gs[1, 1])

    limit = 8.0 
    highlight_red = "#D6604D"
    highlight_blue = "#2166AC"
    cmap = LinearSegmentedColormap.from_list("DivBWR", [highlight_red, "#FFFFFF", highlight_blue], N=256)
    norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)

    sns.heatmap(matrix, ax=ax_heat, cmap=cmap, norm=norm,
                cbar=True, cbar_ax=ax_cbar, mask=mask)

    ax_struc.set_xlim(0, display_limit)
    ax_struc.set_ylim(0, 1)
    
    color_utr = '#E0E0E0'
    color_cds = '#00897B' # 与图1注意力分布中的 CDS 颜色保持一致
    
    if cds_start > 0:
        ax_struc.add_patch(patches.Rectangle((0, 0), cds_start, 1, linewidth=0, facecolor=color_utr))
        ax_struc.text(cds_start - 10, 0.5, "5' UTR", ha='right', va='center', 
                      fontsize=13, color='#333333')

    ax_struc.add_patch(patches.Rectangle((cds_start, 0), cds_end - cds_start, 1, linewidth=0, facecolor=color_cds))
    ax_struc.text((cds_start + cds_end)/2, 0.5, "CDS (Coding Sequence)", ha='center', va='center', 
                  fontsize=14, color='white')

    if cds_end < display_limit:
        ax_struc.add_patch(patches.Rectangle((cds_end, 0), display_limit - cds_end, 1, linewidth=0, facecolor=color_utr))
        ax_struc.text((cds_end + display_limit)/2, 0.5, "3' UTR", ha='center', va='center', 
                      fontsize=13, color='#333333')
    
    ax_struc.axis('off')

    ax_heat.set_xlim(0, display_limit)

    for ax in [ax_struc, ax_heat]:
        ax.axvline(x=cds_start, color='#555555', linestyle='--', linewidth=1.2, alpha=0.5)
        ax.axvline(x=cds_end, color='#555555', linestyle='--', linewidth=1.2, alpha=0.5)

    ax_heat.set_xlabel("Position in Sequence (Nucleotide Index)", fontsize=17, labelpad=12)
    ax_heat.set_ylabel("Mutated Base", fontsize=17, labelpad=8)
    
    ax_heat.set_yticklabels(ax_heat.get_yticklabels(), rotation=0, fontsize=14)
    
    xticks = np.arange(0, display_limit, 200)
    ax_heat.set_xticks(xticks)
    ax_heat.set_xticklabels(xticks, fontsize=14, rotation=45, ha='right') 

    for _, spine in ax_heat.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.2)
        spine.set_color('black')

    ax_cbar.tick_params(labelsize=13)
    ax_cbar.set_ylabel('$\Delta$ Perplexity (Mut - WT)', fontsize=17, labelpad=12)
    for spine in ax_cbar.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    ax_heat.set_facecolor('#FAFAFA')
    plt.tight_layout()
    plt.show()

plot_mutation_heatmap_final_polished(df, text)


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['xtick.major.width'] = 1.2
plt.rcParams['ytick.major.width'] = 1.2
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'

def parse_gene_structure_exact(full_text):
    s_tag = "<gene>"
    e_tag = "</gene>"
    s_idx = full_text.find(s_tag)
    e_idx = full_text.find(e_tag)
    if s_idx == -1 or e_idx == -1: return len(full_text), 0, 0
    
    gene_content = full_text[s_idx + len(s_tag) : e_idx]
    pure_len = 0
    cds_start, cds_end = None, None
    
    i = 0
    while i < len(gene_content):
        if gene_content[i] == "<":
            j = gene_content.find(">", i)
            if j != -1:
                tag = gene_content[i : j+1]
                if tag == "<CDS>": cds_start = pure_len
                elif tag == "</CDS>": cds_end = pure_len
                i = j + 1
                continue
        if gene_content[i].strip(): pure_len += 1
        i += 1
    return pure_len, cds_start, cds_end

def plot_mutation_heatmap_zoom_start_codon_final(df, full_text, window_start=100, window_end=150):
    total_len, cds_start, cds_end = parse_gene_structure_exact(full_text)
    if cds_start is None: cds_start, cds_end = 100, 200

    tmp = df[(df["pos_in_gene_nt"] >= window_start) & (df["pos_in_gene_nt"] < window_end)].copy()
    if tmp.empty:
        print(f"No data found in window {window_start}-{window_end}")
        return

    bases = ["A", "C", "G", "T", "DEL"]
    matrix = tmp.pivot_table(
        index="mut_base", columns="pos_in_gene_nt", values="delta_ppl"
    ).reindex(bases)
    
    expected_cols = np.arange(window_start, window_end)
    matrix = matrix.reindex(columns=expected_cols)

    mask = np.zeros_like(matrix, dtype=bool)
    ref_series = tmp.groupby("pos_in_gene_nt")["ref_base"].first()
    for col_pos in expected_cols:
        if col_pos in ref_series.index:
            ref_b = ref_series[col_pos]
            if ref_b in bases:
                row_idx = bases.index(ref_b)
                col_idx = list(expected_cols).index(col_pos)
                mask[row_idx, col_idx] = True

    fig = plt.figure(figsize=(10, 6.5), dpi=300)
    gs = gridspec.GridSpec(2, 2, width_ratios=[40, 1], height_ratios=[0.5, 10], 
                           wspace=0.015, hspace=0.03)

    ax_struc = fig.add_subplot(gs[0, 0])
    ax_heat = fig.add_subplot(gs[1, 0])
    ax_cbar = fig.add_subplot(gs[1, 1])

    limit = 4.0 
    highlight_red = "#D6604D"
    highlight_blue = "#2166AC"
    cmap = LinearSegmentedColormap.from_list("DivBWR", [highlight_red, "#FFFFFF", highlight_blue], N=256)
    norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)

    sns.heatmap(matrix, ax=ax_heat, cmap=cmap, norm=norm,
                cbar=True, cbar_ax=ax_cbar, mask=mask)

    ax_struc.set_xlim(window_start, window_end)
    ax_struc.set_ylim(0, 1)
    
    color_utr = '#E0E0E0'
    color_cds = '#00897B'
    
    if cds_start > window_start:
        w_s = window_start
        w_e = min(cds_start, window_end)
        ax_struc.add_patch(patches.Rectangle((w_s, 0), w_e - w_s, 1, linewidth=0, facecolor=color_utr))
        if w_e - w_s > 5:
            ax_struc.text((w_s + w_e)/2, 0.5, "5' UTR", ha='center', va='center', fontsize=13, color='#333333')

    cds_s_draw = max(cds_start, window_start)
    cds_e_draw = min(cds_end, window_end)
    if cds_e_draw > cds_s_draw:
        ax_struc.add_patch(patches.Rectangle((cds_s_draw, 0), cds_e_draw - cds_s_draw, 1, linewidth=0, facecolor=color_cds))
        ax_struc.text((cds_s_draw + cds_e_draw)/2, 0.5, "CDS", ha='center', va='center', fontsize=14, color='white')

    ax_struc.axis('off')

    target_start = 119
    original_red = "#E64B35" # 还原为你的箱线图和原始红色
    
    if window_start <= target_start < window_end:
        rect_x = target_start - window_start
        rect_width = 3 
        rect_height = 5 
        
        rect = patches.Rectangle((rect_x, 0), rect_width, rect_height, 
                                 linewidth=2.5, edgecolor=original_red, facecolor='none', zorder=10)
        ax_heat.add_patch(rect)
        
        ax_heat.text(rect_x + rect_width + 0.5, 2.5, "Start Codon", 
                     ha='left', va='center', fontsize=20, fontweight='bold', color=original_red)

    ax_heat.set_xlabel("Position in Sequence (Nucleotide Index)", fontsize=17, labelpad=12)
    ax_heat.set_ylabel("Mutated Base", fontsize=17, labelpad=8)
    ax_heat.set_yticklabels(ax_heat.get_yticklabels(), rotation=0, fontsize=14)
    
    tick_step = 5
    start_tick = (window_start // tick_step + 1) * tick_step
    if start_tick < window_start: start_tick += tick_step
    
    custom_ticks = np.arange(start_tick, window_end, tick_step)
    tick_locs = [t - window_start + 0.5 for t in custom_ticks]
    ax_heat.set_xticks(tick_locs)
    ax_heat.set_xticklabels(custom_ticks, fontsize=14, rotation=0)

    for _, spine in ax_heat.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.2)
        spine.set_color('black')

    ax_cbar.tick_params(labelsize=14)
    ax_cbar.set_ylabel('$\Delta$ Perplexity (Mut - WT)', fontsize=17, labelpad=12)
    for spine in ax_cbar.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    ax_heat.set_facecolor('#FAFAFA')

    plt.tight_layout()
    plt.show()

plot_mutation_heatmap_zoom_start_codon_final(df, text, window_start=100, window_end=150)


In [ ]:
CODON_TABLE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
    'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
}

def get_region_indices(gene_seq_with_tags):
    """
    
    Args:
        gene_seq_with_tags: 包含 <CDS> 等标签的字符串 (即 extract_gene_region 的中间部分)
    Returns:
        cds_start_idx: int, CDS 第一个碱基在去标签序列中的索引
        cds_end_idx: int, CDS 最后一个碱基的下一个位置索引
        pure_seq: str, 去除标签后的纯序列
    """
    pure_seq = []
    cds_start_idx = None
    cds_end_idx = None
    
    current_idx = 0
    i = 0
    n = len(gene_seq_with_tags)
    
    while i < n:
        if gene_seq_with_tags[i] == '<':
            j = gene_seq_with_tags.find('>', i)
            if j == -1: break
            tag = gene_seq_with_tags[i:j+1]
            
            if tag == "<CDS>":
                cds_start_idx = current_idx
            elif tag == "</CDS>":
                cds_end_idx = current_idx
            
            i = j + 1 # 跳过标签
        else:
            if gene_seq_with_tags[i] in "ACGTacgt":
                pure_seq.append(gene_seq_with_tags[i])
                current_idx += 1
            i += 1
            
    return cds_start_idx, cds_end_idx, "".join(pure_seq)

def translate_dna(dna_seq):
    protein = []
    dna_seq = dna_seq.upper()
    for i in range(0, len(dna_seq) - 2, 3):
        codon = dna_seq[i:i+3]
        if len(codon) == 3:
            protein.append(CODON_TABLE.get(codon, 'X'))
    return "".join(protein)

def classify_mutation(row, cds_start, cds_end, full_ref_seq):
    pos = row['pos_in_gene_nt']
    mut_type = row['mutation_type']
    mut_base = row['mut_base']
    ref_base = row['ref_base']
    
    if pos < cds_start:
        region = "5_UTR"
    elif pos >= cds_end:
        region = "3_UTR"
    else:
        region = "CDS"
        
    if region == "5_UTR":
        if mut_type == "deletion":
            return "5' UTR Deletion"
        else:
            return "5' UTR Substitution"
            
    elif region == "3_UTR":
        if mut_type == "deletion":
            return "3' UTR Deletion"
        else:
            return "3' UTR Substitution"
            
    elif region == "CDS":
        if mut_type == "deletion":
            return "CDS Deletion (Frameshift)"
        else:
            rel_pos = pos - cds_start
            codon_idx = rel_pos // 3
            offset_in_codon = rel_pos % 3
            
            cds_seq = full_ref_seq[cds_start:cds_end].upper()
            
            start_codon_pos = codon_idx * 3
            original_codon = list(cds_seq[start_codon_pos : start_codon_pos+3])
            
            if len(original_codon) < 3:
                return "CDS Indeterminate"

            orig_aa = CODON_TABLE.get("".join(original_codon), "X")
            
            mutated_codon = original_codon.copy()
            mutated_codon[offset_in_codon] = mut_base.upper()
            mut_aa = CODON_TABLE.get("".join(mutated_codon), "X")
            
            if mut_aa == orig_aa:
                return "CDS Synonymous"
            elif mut_aa == "*":
                return "CDS Nonsense" # 提前终止
            elif orig_aa == "*":
                return "CDS Stop-Loss" # 终止密码子突变为氨基酸
            else:
                return "CDS Missense" # 错义突变

    return "Unknown"

def analyze_mutation_statistics(df, raw_gene_text):
    """
    主分析函数：接收原始 df 和含标签的 gene 文本，返回统计结果。
    """
    cds_start, cds_end, pure_seq = get_region_indices(raw_gene_text)
    
    if cds_start is None or cds_end is None:
        print("Warning: <CDS> tags not found correctly. Calculating pure stats only.")
        return None
    
    print(f"Structure Parsed: CDS Start (nt index): {cds_start}, CDS End: {cds_end}")
    print(f"CDS Length: {cds_end - cds_start} bp")
    
    df['category'] = df.apply(
        lambda row: classify_mutation(row, cds_start, cds_end, pure_seq), 
        axis=1
    )
    
    stats = df.groupby('category')['delta_ppl'].agg(['mean', 'var', 'count', 'std'])
    
    stats.columns = ['Mean ΔPPL', 'Variance', 'Count', 'Std Dev']
    
    return stats, df


def extract_gene_content(full_text):
    start_tag = "<gene>"
    end_tag = "</gene>"
    s = full_text.find(start_tag)
    e = full_text.find(end_tag)
    if s != -1 and e != -1:
        return full_text[s+len(start_tag):e]
    return ""





In [ ]:
gene_seq_content = extract_gene_region(text)[1]

stats_df, labeled_df = analyze_mutation_statistics(df, gene_seq_content)

print("=== Mutation impact summary (ΔPPL) ===")
display(stats_df.sort_values("Mean ΔPPL", ascending=False))

print("=== CDS nonsense mutation examples ===")
display(labeled_df[labeled_df["category"] == "CDS Nonsense"].head())
